In [ ]:
# 02_collaborative_filtering.ipynb

## 0. Contributors
## 1. Objective
## 2. Data Loading
## 3. Data Preparation
## 4. Train/Test Split
## 5. User-Item Matrix Construction
## 6. Item-Based Collaborative Filtering
## 7. Recommendation Generation
## 8. Offline Evaluation
## 9. Final Insights

#**0. Contributors**
- Jacobo Galindo Sanz

#**1. Objective**

The objective of this notebook is to implement an item-based collaborative filtering recommender that captures user-specific preferences from past interactions. This approach aims to move beyond global popularity by generating personalized product recommendations based on similarity patterns between items.


#**2. Imports and Data Loading**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

In [ ]:
# Load only required columns first (no strict dtypes yet)
orders = pd.read_csv(
    "orders.csv",
    usecols=["order_id", "user_id", "eval_set", "order_number"],
    low_memory=False
)

order_products_prior = pd.read_csv(
    "order_products__prior.csv",
    usecols=["order_id", "product_id", "reordered"],
    low_memory=False
)

products = pd.read_csv(
    "products.csv",
    usecols=["product_id", "product_name"],
    low_memory=False
)

# Convert types safely AFTER loading
orders["order_id"] = pd.to_numeric(orders["order_id"], errors="coerce")
orders["user_id"] = pd.to_numeric(orders["user_id"], errors="coerce")
orders["order_number"] = pd.to_numeric(orders["order_number"], errors="coerce")

order_products_prior["order_id"] = pd.to_numeric(order_products_prior["order_id"], errors="coerce")
order_products_prior["product_id"] = pd.to_numeric(order_products_prior["product_id"], errors="coerce")
order_products_prior["reordered"] = pd.to_numeric(order_products_prior["reordered"], errors="coerce")

products["product_id"] = pd.to_numeric(products["product_id"], errors="coerce")

# Drop bad rows
orders = orders.dropna(subset=["order_id", "user_id", "order_number"]).copy()
order_products_prior = order_products_prior.dropna(subset=["order_id", "product_id", "reordered"]).copy()
products = products.dropna(subset=["product_id"]).copy()

# Final lightweight types
orders["order_id"] = orders["order_id"].astype("int32")
orders["user_id"] = orders["user_id"].astype("int32")
orders["order_number"] = orders["order_number"].astype("int16")
orders["eval_set"] = orders["eval_set"].astype("category")

order_products_prior["order_id"] = order_products_prior["order_id"].astype("int32")
order_products_prior["product_id"] = order_products_prior["product_id"].astype("int32")
order_products_prior["reordered"] = order_products_prior["reordered"].astype("int8")

products["product_id"] = products["product_id"].astype("int32")

print("orders:", orders.shape)
print("order_products_prior:", order_products_prior.shape)
print("products:", products.shape)

orders: (1966867, 4)
order_products_prior: (4590903, 3)
products: (49688, 2)


#**3. Data Preparation**


In [ ]:
# Keep only prior orders
orders_prior = orders[orders["eval_set"] == "prior"][["order_id", "user_id", "order_number"]].copy()

# Merge interactions
prior = order_products_prior.merge(
    orders_prior,
    on="order_id",
    how="inner"
)

print("prior shape:", prior.shape)
prior.head()

prior shape: (2642727, 5)


,order_id,product_id,reordered,user_id,order_number
0,6,40462,0,22352,4
1,6,15873,0,22352,4
2,6,41897,0,22352,4
3,8,23423,1,3107,5
4,13,17330,0,45082,2


In [ ]:
# Free memory
del orders
del order_products_prior

import gc
gc.collect()

0

In [ ]:
product_lookup = products.copy()
product_lookup.head()

,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...
4,5,Green Chile Anytime Sauce


In [ ]:
del products
gc.collect()

31

In [ ]:
n_users = prior["user_id"].nunique()
n_products = prior["product_id"].nunique()
n_orders = prior["order_id"].nunique()
n_interactions = len(prior)

print("Users:", n_users)
print("Products:", n_products)
print("Orders:", n_orders)
print("Interactions:", n_interactions)

sparsity = 1 - (n_interactions / (n_users * n_products))
print(f"Sparsity: {sparsity:.2%}")

Users: 86121
Products: 42027
Orders: 262019
Interactions: 2642727
Sparsity: 99.93%


#**4. Train/Test Split**


In [ ]:
# Safe train/test split based on last order per user

prior_split = prior.copy()

if "last_order_number" in prior_split.columns:
    prior_split = prior_split.drop(columns=["last_order_number"])

user_last_order = (
    prior_split.groupby("user_id")["order_number"]
    .max()
    .reset_index()
)

user_last_order.columns = ["user_id", "last_order_number"]

prior_split = prior_split.merge(user_last_order, on="user_id", how="left")

train_df = prior_split[prior_split["order_number"] < prior_split["last_order_number"]].copy()
test_df = prior_split[prior_split["order_number"] == prior_split["last_order_number"]].copy()

# Keep only users present in train
valid_users = train_df["user_id"].unique()
test_df = test_df[test_df["user_id"].isin(valid_users)].copy()

# Keep only users present in both train and test
train_users = set(train_df["user_id"].unique())
test_users = set(test_df["user_id"].unique())
common_users = train_users.intersection(test_users)

train_df = train_df[train_df["user_id"].isin(common_users)].copy()
test_df = test_df[test_df["user_id"].isin(common_users)].copy()

print("Train interactions:", len(train_df))
print("Test interactions:", len(test_df))
print("Train users:", train_df["user_id"].nunique())
print("Test users:", test_df["user_id"].nunique())

Train interactions: 1761665
Test interactions: 570726
Train users: 54550
Test users: 54550


In [ ]:
ground_truth = (
    test_df.groupby("user_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("Users in ground truth:", len(ground_truth))

Users in ground truth: 54550


#**5. User-Item Matrix Construction**


In [ ]:
# Keep only top products to control RAM and improve stability

top_n_products = 1500

top_products = (
    train_df["product_id"]
    .value_counts()
    .head(top_n_products)
    .index
)

train_cf = train_df[train_df["product_id"].isin(top_products)].copy()
test_cf = test_df[test_df["product_id"].isin(top_products)].copy()

print("Train interactions after filtering:", len(train_cf))
print("Test interactions after filtering:", len(test_cf))
print("Products kept:", train_cf["product_id"].nunique())
print("Users kept:", train_cf["user_id"].nunique())

Train interactions after filtering: 1089361
Test interactions after filtering: 343730
Products kept: 1500
Users kept: 52895


In [ ]:
# Map IDs to matrix indices
user_ids = train_cf["user_id"].unique()
product_ids = train_cf["product_id"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
product_to_idx = {p: i for i, p in enumerate(product_ids)}
idx_to_product = {i: p for p, i in product_to_idx.items()}

# Sparse binary interaction matrix
rows = train_cf["user_id"].map(user_to_idx).values
cols = train_cf["product_id"].map(product_to_idx).values
data = np.ones(len(train_cf), dtype=np.float32)

user_item_sparse = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(product_ids))
)

print("Sparse matrix shape:", user_item_sparse.shape)
print("Non-zero entries:", user_item_sparse.nnz)

Sparse matrix shape: (52895, 1500)
Non-zero entries: 753452


In [ ]:
matrix_sparsity = 1 - (user_item_sparse.nnz / (user_item_sparse.shape[0] * user_item_sparse.shape[1]))
print(f"Filtered matrix sparsity: {matrix_sparsity:.2%}")

Filtered matrix sparsity: 99.05%


#**6. Item-Based Collaborative Filtering**


In [ ]:
# Item-user sparse matrix
item_user_sparse = user_item_sparse.T.tocsr()

# Normalize item vectors for cosine similarity
item_user_norm = normalize(item_user_sparse, axis=1)

print("Item-user matrix shape:", item_user_norm.shape)

Item-user matrix shape: (1500, 52895)


In [ ]:
user_history = (
    train_cf.groupby("user_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("Users with purchase history:", len(user_history))

Users with purchase history: 52895


In [ ]:
def recommend_item_based(user_id, k=10, top_similar_per_item=30):
    if user_id not in user_history:
        return []

    purchased = user_history[user_id]
    scores = {}

    for item in purchased:
        if item not in product_to_idx:
            continue

        item_idx = product_to_idx[item]

        # Sparse cosine similarity with all items
        sims = item_user_norm @ item_user_norm[item_idx].T
        sims = np.asarray(sims.toarray()).ravel()

        # Remove self-similarity
        sims[item_idx] = 0.0

        # Top similar items only
        top_idx = np.argpartition(sims, -top_similar_per_item)[-top_similar_per_item:]

        for idx in top_idx:
            candidate = idx_to_product[idx]
            if candidate in purchased:
                continue
            scores[candidate] = scores.get(candidate, 0.0) + sims[idx]

    if not scores:
        return []

    ranked_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [item for item, score in ranked_items[:k]]

#**7. Recommendation Generation**


In [ ]:
sample_user = list(user_history.keys())[0]
sample_recommendations = recommend_item_based(sample_user, k=10)

print("Sample user:", sample_user)
print("Recommended product IDs:", sample_recommendations)

Sample user: 1
Recommended product IDs: [np.int32(38928), np.int32(6184), np.int32(46149), np.int32(39657), np.int32(31651), np.int32(37710), np.int32(45051), np.int32(13575), np.int32(42500), np.int32(22362)]


In [ ]:
sample_rec_df = pd.DataFrame({"product_id": sample_recommendations})
sample_rec_df = sample_rec_df.merge(product_lookup, on="product_id", how="left")
sample_rec_df

,product_id,product_name
0,38928,0% Greek Strained Yogurt
1,6184,Clementines
2,46149,Zero Calorie Cola
3,39657,Milk Chocolate Almonds
4,31651,Extra Fancy Unsalted Mixed Nuts
5,37710,Trail Mix
6,45051,Pub Mix
7,13575,Apples
8,42500,Orange & Lemon Flavor Variety Pack Sparkling F...
9,22362,Original Rice Krispies Treats


#**8. Offline Evaluation**


In [ ]:
ground_truth_cf = (
    test_cf.groupby("user_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("Users with test items in filtered catalog:", len(ground_truth_cf))

Users with test items in filtered catalog: 51426


In [ ]:
import math
import numpy as np

def precision_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    if len(recommended) == 0:
        return 0.0
    return len(set(recommended) & set(relevant)) / k

def recall_at_k(recommended, relevant, k=10):
    if len(relevant) == 0:
        return 0.0
    recommended = recommended[:k]
    return len(set(recommended) & set(relevant)) / len(relevant)

def ndcg_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    dcg = 0.0
    for i, item in enumerate(recommended, start=1):
        if item in relevant:
            dcg += 1 / math.log2(i + 1)

    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0

    idcg = sum(1 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg

In [ ]:
sample_users = list(ground_truth_cf.keys())[:500]

cf_results = []

for user_id in sample_users:
    relevant_items = ground_truth_cf[user_id]
    recs = recommend_item_based(user_id, k=10)

    cf_results.append({
        "precision_cf": precision_at_k(recs, relevant_items, 10),
        "recall_cf": recall_at_k(recs, relevant_items, 10),
        "ndcg_cf": ndcg_at_k(recs, relevant_items, 10),
        "recommended": recs
    })

cf_results_df = pd.DataFrame(cf_results)
cf_results_df.head()

,precision_cf,recall_cf,ndcg_cf,recommended
0,0.0,0.000000,0.000000,"[38928, 6184, 46149, 39657, 31651, 37710, 4505..."
1,0.1,1.000000,1.000000,"[21137, 24852, 13176, 21903, 47209, 24964, 262..."
2,0.0,0.000000,0.000000,"[21903, 21137, 47209, 24964, 13176, 26209, 248..."
3,0.1,1.000000,0.386853,"[24852, 21137, 13176, 47209, 16797, 27966, 827..."
4,0.2,0.222222,0.192157,"[4957, 33787, 40571, 24852, 21137, 22124, 1915..."


In [ ]:
catalog_size = train_cf["product_id"].nunique()

recommended_items = set()
for recs in cf_results_df["recommended"]:
    recommended_items.update(recs)

coverage_cf = len(recommended_items) / catalog_size

In [ ]:
def precision_at_k(recommended, relevant, k=10):
    recommended_k = recommended[:k]
    if len(recommended_k) == 0:
        return 0.0
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k=10):
    if len(relevant) == 0:
        return 0.0
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant)

In [ ]:
def simple_diversity_at_k(recommended, k=10):
    return len(set(recommended[:k])) / k

diversity_cf = np.mean([
    simple_diversity_at_k(recs)
    for recs in cf_results_df["recommended"]
])

In [ ]:
popular_set = set(
    train_df["product_id"].value_counts().head(20).index
)

def serendipity_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    hits = [item for item in recommended if item in relevant and item not in popular_set]
    return len(hits) / k

serendipity_cf = np.mean([
    serendipity_at_k(row["recommended"], ground_truth_cf[sample_users[i]])
    for i, row in cf_results_df.iterrows()
])

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

test_eval_cf = test_cf.copy()
test_eval_cf["pred"] = 0.5  # simple proxy

y_true = test_eval_cf["reordered"]

rmse_cf = np.sqrt(mean_squared_error(y_true, test_eval_cf["pred"]))
mae_cf = mean_absolute_error(y_true, test_eval_cf["pred"])

#**9. Final Insights**


In [ ]:
comparison_table_cf = pd.DataFrame([
    {
        "Approach": "Item-Based CF",
        "RMSE": rmse_cf,
        "MAE": mae_cf,
        "Precision@K": cf_results_df["precision_cf"].mean(),
        "Recall@K": cf_results_df["recall_cf"].mean(),
        "NDCG": cf_results_df["ndcg_cf"].mean(),
        "Coverage": coverage_cf,
        "Diversity": diversity_cf,
        "Serendipity": serendipity_cf,
        "Context": "Personalized item similarity model"
    }
])

comparison_table_cf.round(4)

,Approach,RMSE,MAE,Precision@K,Recall@K,NDCG,Coverage,Diversity,Serendipity,Context
0,Item-Based CF,0.5,0.5,0.0378,0.0557,0.0594,0.252,0.97,0.0054,Personalized item similarity model


#**EXTRA: Recommendation Example For The Slides**


In [ ]:
# Build product_id -> product_name lookup
product_name_map = (
    product_lookup
    .drop_duplicates(subset=["product_id"])
    .set_index("product_id")["product_name"]
    .to_dict()
)

In [ ]:
# Function to display one real user example
def show_user_example(user_id, history_limit=8, k=5):
    purchased_ids = list(user_history.get(user_id, []))
    recommended_ids = recommend_item_based(user_id, k=k)

    print(f"USER ID: {user_id}")

    print("\nPast Purchases:")
    for pid in purchased_ids[:history_limit]:
        print("-", product_name_map.get(pid, pid))

    print("\nRecommended Products:")
    for pid in recommended_ids:
        print("-", product_name_map.get(pid, pid))

In [ ]:
# Try several real users
sample_users = list(user_history.keys())[:10]

for uid in sample_users:
    print("=" * 50)
    show_user_example(uid, history_limit=8, k=5)
    print()


USER ID: 1

Past Purchases:
- Creamy Almond Butter
- Original Beef Jerky
- Soda
- Organic String Cheese

Recommended Products:
- 0% Greek Strained Yogurt
- Clementines
- Zero Calorie Cola
- Milk Chocolate Almonds
- Extra Fancy Unsalted Mixed Nuts

USER ID: 7

Past Purchases:
- Large Alfresco Eggs
- Honeycrisp Apple
- 85% Lean Ground Beef
- Mexican Finely Shredded Cheese
- Lactose Free Fat Free Milk
- Apple Cinnamon GoGo Squeez
- Blood Oranges
- Organic Apple Slices

Recommended Products:
- Organic Strawberries
- Banana
- Bag of Organic Bananas
- Organic Baby Spinach
- Organic Hass Avocado

USER ID: 10

Past Purchases:
- Organic Broccoli Florets
- Asparagus
- Organic Romaine Lettuce
- Sliced Baby Bella Mushrooms
- Bing Cherries
- Organic Ketchup
- 85% Lean Ground Beef
- Organic Turkey Bacon

Recommended Products:
- Organic Baby Spinach
- Organic Strawberries
- Organic Hass Avocado
- Organic Garlic
- Bag of Organic Bananas

USER ID: 17

Past Purchases:
- 100% Whole Wheat Bread
- Strawber

In [ ]:
# Create a neat table for one user
def get_user_example_table(user_id, history_limit=8, k=5):
    purchased_ids = list(user_history.get(user_id, []))
    recommended_ids = recommend_item_based(user_id, k=k)

    purchased_names = [product_name_map.get(pid, pid) for pid in purchased_ids[:history_limit]]
    recommended_names = [product_name_map.get(pid, pid) for pid in recommended_ids]

    max_len = max(len(purchased_names), len(recommended_names))
    purchased_names += [""] * (max_len - len(purchased_names))
    recommended_names += [""] * (max_len - len(recommended_names))

    return pd.DataFrame({
        "Past Purchases": purchased_names,
        "Recommended Products": recommended_names
    })

In [ ]:
# Build clean comparison table for USER ID 21

user_id = 21

# Get purchased + recommended IDs
purchased_ids = list(user_history[user_id])
recommended_ids = recommend_item_based(user_id, k=5)

# Convert IDs to names
purchased_names = [product_name_map.get(pid, pid) for pid in purchased_ids]
recommended_names = [product_name_map.get(pid, pid) for pid in recommended_ids]

# Keep only first rows for presentation
purchased_names = purchased_names[:8]
recommended_names = recommended_names[:8]

# Make lengths equal
max_len = max(len(purchased_names), len(recommended_names))
purchased_names += [""] * (max_len - len(purchased_names))
recommended_names += [""] * (max_len - len(recommended_names))

# Create table
user21_table = pd.DataFrame({
    "Past Purchases": purchased_names,
    "Recommended Products": recommended_names
})

user21_table

,Past Purchases,Recommended Products
0,Total 2% with Strawberry Lowfat Greek Strained...,Total 2% Lowfat Greek Strained Yogurt With Blu...
1,India Pale Ale,Total 2% Lowfat Greek Strained Yogurt with Peach
2,Organic Gala Apples,Total 2% Greek Strained Yogurt with Cherry 5.3 oz
3,Slim Can Pink Grapefruit Natural Mineral Water,Banana
4,Canned Aranciata Orange,Organic Strawberries
5,Plain Pre-Sliced Bagels,
6,Total 2% All Natural Greek Strained Yogurt wit...,
7,Unsweetened Premium Iced Tea,
